In [ ]:
# pip install -U google-genai python-dotenv tqdm pandas pyarrow nest_asyncio

# LLM-based Classification (kununu)
LLM-based classification of German Kununu reviews into Herzberg's 14 two-factor theory categories.

In [ ]:
import sqlite3
from google import genai
from google.genai import types
import os, json, asyncio, random
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
import numpy as np
from tqdm.asyncio import tqdm
import nest_asyncio

nest_asyncio.apply()
load_dotenv()

In [ ]:
API_KEY = os.getenv("GEMINI_API_KEY")

client = genai.Client(api_key=API_KEY)

MODEL             = "gemini-2.5-flash-lite"
BATCH_SIZE        = 20
CONCURRENT_LIMIT  = 30
MAX_RETRIES       = 4
SAVE_INTERVAL     = 600   

DB_PATH = "PJDB.db"

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    conn.execute("""
        CREATE TABLE IF NOT EXISTS llm_proscons_kununu (
            review_id INTEGER PRIMARY KEY,
            p_company_policy_administration INTEGER,
            p_supervision INTEGER,
            p_interpersonal_relations INTEGER,
            p_working_conditions INTEGER,
            p_salary INTEGER,
            p_personal_life INTEGER,
            p_job_security INTEGER,
            p_achievement INTEGER,
            p_recognition INTEGER,
            p_work_itself INTEGER,
            p_responsibility INTEGER,
            p_advancement INTEGER,
            p_possibility_of_growth INTEGER,
            p_status INTEGER,
            c_company_policy_administration INTEGER,
            c_supervision INTEGER,
            c_interpersonal_relations INTEGER,
            c_working_conditions INTEGER,
            c_salary INTEGER,
            c_personal_life INTEGER,
            c_job_security INTEGER,
            c_achievement INTEGER,
            c_recognition INTEGER,
            c_work_itself INTEGER,
            c_responsibility INTEGER,
            c_advancement INTEGER,
            c_possibility_of_growth INTEGER,
            c_status INTEGER
        )
    """)

n_classified = conn.execute("SELECT COUNT(*) FROM llm_proscons_kununu").fetchone()[0]
print(f"llm_proscons_kununu: {n_classified:,} rows already classified")

### Data Loading

In [4]:
import sqlite3
import pandas as pd
import numpy as np

with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql(
        """SELECT id, company, date, rating, recommendation, position,
                  headline, pros, cons,
                  atmosphere, image, work_life_balance, career, salary,
                  colleagues, management, working_conditions, equal_rights,
                  interesting_tasks, communication, senior_colleagues,
                  environment_social
           FROM REVIEWS_GER""",
        conn,
    )

df["review_id"] = df["id"].astype(int)
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

print(f"Reviews loaded: {len(df):,}")
print(f"Companies:  {df['company'].nunique()}")
print(f"Columns:    {list(df.columns)}")

Reviews loaded: 19,370
Companies:  46
Columns:    ['id', 'company', 'date', 'rating', 'recommendation', 'position', 'headline', 'pros', 'cons', 'atmosphere', 'image', 'work_life_balance', 'career', 'salary', 'colleagues', 'management', 'working_conditions', 'equal_rights', 'interesting_tasks', 'communication', 'senior_colleagues', 'environment_social', 'review_id']


### Classification

In [ ]:
FACTORS_14 = [
    "company_policy_administration", "supervision", "interpersonal_relations",
    "working_conditions", "salary", "personal_life", "job_security",
    "achievement", "recognition", "work_itself", "responsibility",
    "advancement", "possibility_of_growth", "status",
]

PROSCONS_SYSTEM_PROMPT = """
You are a precise text analyst specialized in organizational psychology.
Classify a list of German employee reviews using Herzberg's 14 two-factor theory categories.
The review texts are in German — classify the MEANING, not the language.

=== FACTOR DEFINITIONS ===

company_policy_administration: Company-level organisation, management policies, HR processes,
  bureaucracy, rules, communication structures, delegation of authority.
supervision: The direct supervisor's competence, fairness, behaviour, or willingness to
  delegate. Only applies to the immediate boss relationship.
interpersonal_relations: The social quality of relationships with colleagues, superiors, or
  subordinates — team atmosphere, collegiality, personal connections at work.
working_conditions: Physical environment (space, equipment, tools, noise, safety) and
  workload volume or intensity. Does NOT include work-life balance effects on private life.
salary: All forms of compensation: base pay, bonuses, benefits, raises.
personal_life: Work's impact on time and life OUTSIDE work — work-life balance, flexible
  hours, remote work, family-friendliness. Does NOT include workload volume itself.
job_security: Stability of the position, contract type, fear of layoffs, company financial
  health.
achievement: The employee's own sense of accomplishing something — completing a challenge,
  solving a problem, reaching a goal. Internal feeling of success.
recognition: External acknowledgement by others — praise, awards, feeling valued or ignored,
  criticism, blame. Requires another person doing the recognising.
work_itself: Intrinsic interest, variety, meaningfulness, or creativity of the actual tasks
  performed.
responsibility: Autonomy, ownership, freedom to make decisions, accountability for one's
  own work area.
advancement: Actual or potential movement up the career ladder — promotions, title changes,
  career progression within the company.
possibility_of_growth: Opportunities to learn and develop personally — training, new skills,
  courses, personal development programmes, access to education.
status: External prestige signals — company brand recognition, impressive job title, rank
  symbols, working for a well-known organisation.

=== DISAMBIGUATION RULES ===

advancement vs possibility_of_growth:
  "Ich wurde befördert" / "keine Karrierechancen" → advancement
  "Ich habe viel gelernt" / "keine Weiterbildung" → possibility_of_growth
  A review can trigger BOTH if it mentions promotion AND learning.

supervision vs interpersonal_relations:
  Direct boss managing, evaluating, or delegating → supervision
  Team spirit, colleague relationships, social atmosphere → interpersonal_relations
  "Mein Chef ist super, aber das Team ist toxisch" → supervision=1 AND interpersonal_relations=1

working_conditions vs personal_life:
  Heavy workload, bad equipment, noisy office → working_conditions
  "Keine Zeit für die Familie", flexible hours, work-life balance → personal_life

recognition vs achievement:
  "Ich bin stolz auf mein Projekt" (internal feeling) → achievement
  "Mein Chef lobt nie jemanden" (someone else acknowledging) → recognition
  "Ich habe ein Problem gelöst und wurde dafür gelobt" → BOTH can be 1

=== MULTI-LABEL RULE ===

A review text can — and often will — match multiple factors simultaneously.
Label ALL factors that are clearly present. Do not try to pick just one.

=== WHAT COUNTS AS A MENTION ===

Label a factor 1 for direct mentions AND clear unambiguous references.
Do NOT label factors that require guessing or reading between the lines.
Detect irony and sarcasm and label the intended meaning accordingly.

=== INPUT / OUTPUT FORMAT ===

Input: a JSON array. Each element has "id", "pro" (positive aspects in German), and
  "con" (negative aspects in German). Empty strings mean the field was not filled.
Output: a single valid JSON object — no arrays, no markdown, no extra text.
  Keys = string review IDs.
  Each value has exactly "pro" and "con" keys.
  Each maps all 14 factor names to 0 or 1.

Example input:
[
  {"id": "42",
   "pro": "Tolles Gehalt und das Team ist wie eine Familie. Viel beruflich gelernt.",
   "con": "Mein Chef kontrolliert alles und HR-Prozesse sind ein Alptraum."}
]

Example output:
{
  "42": {
    "pro": {"company_policy_administration": 0, "supervision": 0, "interpersonal_relations": 1,
            "working_conditions": 0, "salary": 1, "personal_life": 0, "job_security": 0,
            "achievement": 0, "recognition": 0, "work_itself": 0, "responsibility": 0,
            "advancement": 0, "possibility_of_growth": 1, "status": 0},
    "con": {"company_policy_administration": 1, "supervision": 1, "interpersonal_relations": 0,
            "working_conditions": 0, "salary": 0, "personal_life": 0, "job_security": 0,
            "achievement": 0, "recognition": 0, "work_itself": 0, "responsibility": 0,
            "advancement": 0, "possibility_of_growth": 0, "status": 0}
  }
}
"""

PROSCONS_GEN_CONFIG = types.GenerateContentConfig(
    system_instruction=PROSCONS_SYSTEM_PROMPT,
    response_mime_type="application/json",
    temperature=0,  # deterministic
)

print(f"Factors: {len(FACTORS_14)}")
print(f"Output columns total: {len(FACTORS_14) * 2} (p_ + c_)")

In [ ]:
def parse_batch_response(text):
    """Parse Gemini JSON response, handles common malformed outputs."""
    try:
        result = json.loads(text)
    except json.JSONDecodeError:
        try:
            result, _ = json.JSONDecoder().raw_decode(text.strip())
        except json.JSONDecodeError as e:
            raise ValueError(f"JSON not parseable: {e}") from e

    if isinstance(result, list):
        normalized = {}
        for item in result:
            rid = str(item.get("id", ""))
            if not rid:
                continue
            normalized[rid] = {
                "pro": item.get("pro") or item.get("pros") or {},
                "con": item.get("con") or item.get("cons") or {},
            }
        result = normalized
    return result


def get_section(data, section):
    alt = section + "s"  # 'pro' -> 'pros', 'con' -> 'cons'
    return data.get(section) or data.get(alt) or {}

In [ ]:
async def process_batch_proscons(semaphore, lock, results_store, rows, pbar):
    async with semaphore:
        batch_input = [
            {
                "id": str(r["review_id"]),
                "pro": str(r["pros"] or "")[:2000],
                "con": str(r["cons"] or "")[:2000],
            }
            for r in rows
        ]
        prompt = json.dumps(batch_input, ensure_ascii=False)

        for attempt in range(MAX_RETRIES):
            try:
                response = await client.aio.models.generate_content(
                    model=MODEL,
                    contents=prompt,
                    config=PROSCONS_GEN_CONFIG,
                )
                if not response.text:
                    raise ValueError("Empty response from model")

                batch_results = parse_batch_response(response.text)

                new_rows = []
                for review_id, data in batch_results.items():
                    pro_data = get_section(data, "pro")
                    con_data = get_section(data, "con")
                    row = {"review_id": review_id}
                    for factor in FACTORS_14:
                        row[f"p_{factor}"] = int(pro_data.get(factor, 0))
                        row[f"c_{factor}"] = int(con_data.get(factor, 0))
                    new_rows.append(row)

                async with lock:
                    results_store.extend(new_rows)

                pbar.update(len(rows))
                return

            except Exception as e:
                if attempt < MAX_RETRIES - 1:
                    wait = (2 ** attempt) + random.uniform(0, 1)
                    print(f"\nAttempt {attempt+1}/{MAX_RETRIES} failed: {e}. Waiting {wait:.1f}s...")
                    await asyncio.sleep(wait)
                else:
                    print(f"\nBatch permanently failed after {MAX_RETRIES} attempts: {e}")

In [ ]:
async def main_proscons():
    with sqlite3.connect(DB_PATH) as conn:
        done_ids = set(
            r[0] for r in conn.execute("SELECT review_id FROM llm_proscons_kununu").fetchall()
        )

    pending = df[
        ~df["review_id"].isin(done_ids)
        & (df["pros"].notna() | df["cons"].notna())
    ].to_dict("records")

    print(f"Already classified: {len(done_ids):,}  |  Remaining: {len(pending):,}")
    if not pending:
        print("Nothing to do.")
        return

    semaphore     = asyncio.Semaphore(CONCURRENT_LIMIT)
    lock          = asyncio.Lock()
    results_store = []
    pbar          = tqdm(total=len(pending), desc="Pros/Cons")

    chunk_size = BATCH_SIZE * CONCURRENT_LIMIT
    for chunk_start in range(0, len(pending), chunk_size):
        chunk   = pending[chunk_start : chunk_start + chunk_size]
        batches = [chunk[i : i + BATCH_SIZE] for i in range(0, len(chunk), BATCH_SIZE)]
        await asyncio.gather(
            *[process_batch_proscons(semaphore, lock, results_store, b, pbar) for b in batches]
        )

        if results_store:
            cols   = ["review_id"] + [f"p_{f}" for f in FACTORS_14] + [f"c_{f}" for f in FACTORS_14]
            ph     = ", ".join(["?"] * len(cols))
            col_str = ", ".join(cols)
            rows = [[r[c] for c in cols] for r in results_store]
            with sqlite3.connect(DB_PATH) as conn:
                conn.executemany(
                    f"INSERT OR IGNORE INTO llm_proscons_kununu ({col_str}) VALUES ({ph})",
                    rows,
                )
                conn.commit()
            results_store.clear()

        await asyncio.sleep(0.2)

    pbar.close()
    print("Pros/Cons classification complete → DB: llm_proscons_kununu")


asyncio.run(main_proscons())

### Preview

In [ ]:
# Quick preview of llm_proscons_kununu
with sqlite3.connect(DB_PATH) as conn:
    preview = pd.read_sql("SELECT * FROM llm_proscons_kununu LIMIT 5", conn)
    n = pd.read_sql("SELECT COUNT(*) AS n FROM llm_proscons_kununu", conn).iloc[0, 0]

print(f"llm_proscons_kununu: {n:,} rows, {len(preview.columns)} columns")
display(preview.head())


### Post-Processing

Reviews where all 14 factors are coded as 1 and the text is shorter than 15 characters carry no useful information. All factor columns are reset to 0 for these entries.

Condition: `sum(p_*) == 14` AND `len(pros) < 15` → set all `p_*` to 0 (same for cons).

In [ ]:
import sqlite3
import pandas as pd

DB_PATH = "PJDB.db"

FACTORS_14 = [
    "company_policy_administration", "supervision", "interpersonal_relations",
    "working_conditions", "salary", "personal_life", "job_security",
    "achievement", "recognition", "work_itself", "responsibility",
    "advancement", "possibility_of_growth", "status",
]
PROS_COLS = [f"p_{f}" for f in FACTORS_14]
CONS_COLS = [f"c_{f}" for f in FACTORS_14]

# load classifications + original texts
with sqlite3.connect(DB_PATH) as conn:
    df_l = pd.read_sql(
        f"""SELECT l.review_id, r.pros, r.cons,
                   {", ".join(PROS_COLS + CONS_COLS)}
              FROM llm_proscons_kununu l
              JOIN REVIEWS_GER r ON r.id = l.review_id""",
        conn,
    )

# find reviews where all factors = 1 and text < 15 chars
all_pros_1  = (df_l[PROS_COLS] == 1).all(axis=1)
short_pros  = df_l["pros"].fillna("").str.strip().str.len() < 15
fix_pros    = df_l.loc[all_pros_1 & short_pros, ["review_id", "pros"]]

all_cons_1  = (df_l[CONS_COLS] == 1).all(axis=1)
short_cons  = df_l["cons"].fillna("").str.strip().str.len() < 15
fix_cons    = df_l.loc[all_cons_1 & short_cons, ["review_id", "cons"]]

print(f"Affected Pros reviews (all factors=1, len<15): {len(fix_pros)}")
if not fix_pros.empty:
    print(fix_pros.head(10).to_string(index=False))

print(f"\nAffected Cons reviews (all factors=1, len<15): {len(fix_cons)}")
if not fix_cons.empty:
    print(fix_cons.head(10).to_string(index=False))

In [ ]:
# write corrections to DB
pros_ids = fix_pros["review_id"].tolist()
cons_ids = fix_cons["review_id"].tolist()

with sqlite3.connect(DB_PATH) as conn:
    if pros_ids:
        set_clause   = ", ".join(f"{c} = 0" for c in PROS_COLS)
        placeholders = ", ".join("?" * len(pros_ids))
        n = conn.execute(
            f"UPDATE llm_proscons_kununu SET {set_clause} WHERE review_id IN ({placeholders})",
            pros_ids,
        ).rowcount
        print(f"Pros fixed: {n} reviews → all p_* set to 0")
    else:
        print("Pros: no correction needed")

    if cons_ids:
        set_clause   = ", ".join(f"{c} = 0" for c in CONS_COLS)
        placeholders = ", ".join("?" * len(cons_ids))
        n = conn.execute(
            f"UPDATE llm_proscons_kununu SET {set_clause} WHERE review_id IN ({placeholders})",
            cons_ids,
        ).rowcount
        print(f"Cons fixed: {n} reviews → all c_* set to 0")
    else:
        print("Cons: no correction needed")

    conn.commit()